In [1]:
import pandas as pd
import os

print("File size:", os.path.getsize("loan_data.csv"), "bytes")

df = pd.read_csv("loan_data.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

File size: 3610978 bytes
Dataset shape: (45000, 14)
Columns: ['person_age', 'person_gender', 'person_education', 'person_income', 'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score', 'previous_loan_defaults_on_file', 'loan_status']


In [2]:
print("DATASET INFORMATION")
print("-" * 40)

print("Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nTarget Distribution:")
print(df["loan_status"].value_counts())

print("\nData Types:")
print(df.dtypes)

DATASET INFORMATION
----------------------------------------
Shape: (45000, 14)

Missing Values:
person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb_person_cred_hist_length        0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
dtype: int64

Duplicate Rows: 0

Target Distribution:
loan_status
0    35000
1    10000
Name: count, dtype: int64

Data Types:
person_age                        float64
person_gender                      object
person_education                   object
person_income                     float64
person_emp_exp                      int64
person_home_ownership              object
loan_amnt           

In [3]:
# Separate independent and dependent variables

X = df.drop("loan_status", axis=1)
y = df["loan_status"]

print("Input features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget classes:")
print(y.unique())

Input features shape: (45000, 13)
Target shape: (45000,)

Target classes:
[1 0]


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training data: (36000, 13)
Testing data: (9000, 13)

Training target distribution:
loan_status
0    0.777778
1    0.222222
Name: proportion, dtype: float64

Testing target distribution:
loan_status
0    0.777778
1    0.222222
Name: proportion, dtype: float64


In [5]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [12]:
import sklearn

print("Scikit-learn version:", sklearn.__version__)
print("Installation successful!")

Scikit-learn version: 1.9.1
Installation successful!


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training data: (36000, 13)
Testing data: (9000, 13)

Training target distribution:
loan_status
0    0.777778
1    0.222222
Name: proportion, dtype: float64

Testing target distribution:
loan_status
0    0.777778
1    0.222222
Name: proportion, dtype: float64


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Identify numerical and categorical columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Numerical features:", list(numeric_features))
print("Categorical features:", list(categorical_features))
print("Preprocessing pipeline created successfully!")

Numerical features: ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score']
Categorical features: ['person_gender', 'person_education', 'person_home_ownership', 'loan_intent', 'previous_loan_defaults_on_file']
Preprocessing pipeline created successfully!


In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_features = X.select_dtypes(include=["number"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessing ready for all five models!")

Preprocessing ready for all five models!


In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "SVM": LinearSVC(random_state=42, max_iter=5000)
}

trained_models = {}

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)

    trained_models[name] = pipeline

    print(name, "trained successfully!")

Logistic Regression trained successfully!
Decision Tree trained successfully!
KNN trained successfully!
Naive Bayes trained successfully!
SVM trained successfully!


In [22]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

results = []

for name, model in trained_models.items():

    # Predict test data
    y_pred = model.predict(X_test)

    # Get probability or decision scores for AUC
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    # Calculate six evaluation metrics
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_score),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred)
    })

# Create comparison table
results_df = pd.DataFrame(results)

print("MODEL PERFORMANCE COMPARISON")
display(results_df.round(4))

MODEL PERFORMANCE COMPARISON


,Model,Accuracy,AUC,Precision,Recall,F1 Score,MCC
0,Logistic Regression,0.8993,0.9562,0.7888,0.7470,0.7673,0.7036
1,Decision Tree,0.8999,0.8558,0.7738,0.7765,0.7751,0.7108
2,KNN,0.8956,0.9246,0.7909,0.7205,0.7541,0.6891
3,Naive Bayes,0.7356,0.9398,0.4566,0.9990,0.6267,0.5484
4,SVM,0.8994,0.9562,0.7889,0.7475,0.7677,0.7040


In [24]:
import os
import joblib

# Create model folder
os.makedirs("model", exist_ok=True)

# Save all five trained pipelines
model_files = {
    "Logistic Regression": "logistic_regression.pkl",
    "Decision Tree": "decision_tree.pkl",
    "KNN": "knn.pkl",
    "Naive Bayes": "naive_bayes.pkl",
    "SVM": "svm.pkl"
}

for name, filename in model_files.items():
    joblib.dump(trained_models[name], os.path.join("model", filename))
    print(name, "saved successfully!")

# Save labelled test data for Streamlit
test_data = X_test.copy()
test_data["loan_status"] = y_test

test_data.to_csv("test_data.csv", index=False)

print("\nTest data saved successfully!")

Logistic Regression saved successfully!
Decision Tree saved successfully!
KNN saved successfully!
Naive Bayes saved successfully!
SVM saved successfully!

Test data saved successfully!


In [26]:
import os

print("PROJECT FILES")
print("-" * 40)

print("Model files:")
for file in os.listdir("model"):
    print("-", file)

print("\nTest data exists:", os.path.exists("test_data.csv"))
print("Test data shape:", pd.read_csv("test_data.csv").shape)

PROJECT FILES
----------------------------------------
Model files:
- logistic_regression.pkl
- decision_tree.pkl
- knn.pkl
- naive_bayes.pkl
- svm.pkl

Test data exists: True
Test data shape: (9000, 14)


In [28]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score,
    f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)

st.set_page_config(
    page_title="Loan Prediction System",
    page_icon="🏦",
    layout="wide"
)

st.title("🏦 Loan Prediction & Model Comparison")
st.write("Evaluate five machine learning models using loan application test data.")

# Model selection
model_files = {
    "Logistic Regression": "logistic_regression.pkl",
    "Decision Tree": "decision_tree.pkl",
    "KNN": "knn.pkl",
    "Naive Bayes": "naive_bayes.pkl",
    "SVM": "svm.pkl"
}

selected_model = st.selectbox(
    "Select Machine Learning Model",
    list(model_files.keys())
)

# CSV upload
uploaded_file = st.file_uploader(
    "Upload labelled test data (CSV)",
    type=["csv"]
)

if uploaded_file is not None:

    data = pd.read_csv(uploaded_file)

    if "loan_status" not in data.columns:
        st.error("CSV must contain the loan_status target column.")
        st.stop()

    X = data.drop("loan_status", axis=1)
    y = data["loan_status"]

    model_path = os.path.join(
        "model", model_files[selected_model]
    )

    model = joblib.load(model_path)

    y_pred = model.predict(X)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X)[:, 1]
    else:
        y_score = model.decision_function(X)

    # Evaluation metrics
    st.subheader("Model Evaluation Metrics")

    metrics = {
        "Accuracy": accuracy_score(y, y_pred),
        "AUC Score": roc_auc_score(y, y_score),
        "Precision": precision_score(y, y_pred, zero_division=0),
        "Recall": recall_score(y, y_pred, zero_division=0),
        "F1 Score": f1_score(y, y_pred, zero_division=0),
        "MCC Score": matthews_corrcoef(y, y_pred)
    }

    cols = st.columns(3)

    for i, (name, value) in enumerate(metrics.items()):
        cols[i % 3].metric(name, f"{value:.4f}")

    # Confusion matrix
    st.subheader("Confusion Matrix")

    cm = confusion_matrix(y, y_pred)

    fig, ax = plt.subplots()
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Rejected (0)", "Approved (1)"],
        yticklabels=["Rejected (0)", "Approved (1)"],
        ax=ax
    )

    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

    st.pyplot(fig)
    plt.close(fig)

    # Classification report
    st.subheader("Classification Report")
    st.text(classification_report(y, y_pred, zero_division=0))

    st.success("Model evaluation completed successfully!")

else:
    st.info("Upload your labelled test_data.csv file to evaluate the selected model.")

Writing app.py


In [30]:
%%writefile requirements.txt

streamlit
pandas
numpy
scikit-learn
joblib
matplotlib
seaborn

Writing requirements.txt


In [32]:
with open("app.py", "r") as file:
    app_code = file.read()

app_code = app_code.replace(
    "Rejected (0)", "No Default (0)"
).replace(
    "Approved (1)", "Default (1)"
)

with open("app.py", "w") as file:
    file.write(app_code)

print("Confusion matrix labels corrected!")

Confusion matrix labels corrected!


In [34]:
%%writefile README.md

# Loan Default Prediction Using Machine Learning

## 1. Problem Statement

The objective of this project is to develop a machine learning-based loan default prediction system. Five classification algorithms are implemented and evaluated to identify their performance in predicting loan default risk.

## 2. Dataset Description

- Dataset source: Kaggle
- Total instances: 45,000
- Total columns: 14
- Input features: 13
- Target variable: loan_status
- Problem type: Binary Classification

The dataset contains applicant demographic information, income, employment experience, loan details, credit history and previous loan defaults.

Target classes:
- 0: No Default
- 1: Default

## 3. GitHub Repository Link

To be updated after GitHub repository creation.

## 4. Models and Evaluation Metrics

The following five classification algorithms were implemented:

| Model | Accuracy | AUC | Precision | Recall | F1 Score | MCC |
|---|---:|---:|---:|---:|---:|---:|
| Logistic Regression | 0.8993 | 0.9562 | 0.7888 | 0.7470 | 0.7673 | 0.7036 |
| Decision Tree | 0.8999 | 0.8558 | 0.7738 | 0.7765 | 0.7751 | 0.7108 |
| KNN | 0.8956 | 0.9246 | 0.7909 | 0.7205 | 0.7541 | 0.6891 |
| Naive Bayes | 0.7356 | 0.9398 | 0.4566 | 0.9990 | 0.6267 | 0.5484 |
| SVM | 0.8994 | 0.9562 | 0.7889 | 0.7475 | 0.7677 | 0.7040 |

## 5. Model Performance Observations

### Logistic Regression
Achieved 89.93% accuracy and an AUC score of 0.9562, demonstrating strong classification and discrimination performance.

### Decision Tree
Achieved the highest accuracy of 89.99%, F1 score of 0.7751 and MCC of 0.7108 among the five models.

### K-Nearest Neighbors
Achieved 89.56% accuracy and the highest precision of 0.7909, but its recall was comparatively lower.

### Naive Bayes
Achieved very high recall of 0.9990 but relatively low precision of 0.4566, indicating a high number of false-positive predictions.

### Support Vector Machine
Achieved 89.94% accuracy and an AUC score of 0.9562, matching Logistic Regression on AUC.

## 6. Overall Best Model

Decision Tree is the overall best-performing model based on accuracy, F1 score and Matthews Correlation Coefficient.

Logistic Regression and SVM achieved the highest AUC score of 0.9562.

## 7. Project Files

- app.py: Streamlit application
- requirements.txt: Required Python libraries
- test_data.csv: Labelled test dataset
- model/: Saved machine learning pipelines
- README.md: Project documentation

## 8. Streamlit Application

Live application link: To be updated after deployment.

Writing README.md


In [36]:
import os

print("PROJECT STRUCTURE")
print("-" * 40)

for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if d not in [".ipynb_checkpoints", "__pycache__"]]

    level = root.count(os.sep)
    indent = "    " * level

    print(indent + os.path.basename(root) + "/")

    for file in files:
        print(indent + "    " + file)

PROJECT STRUCTURE
----------------------------------------
./
    Loan_Prediction_ML_Assignment_2.ipynb
    loan_data.csv
    test_data.csv
    app.py
    requirements.txt
    README.md
    model/
        logistic_regression.pkl
        decision_tree.pkl
        knn.pkl
        naive_bayes.pkl
        svm.pkl
